## DOUAE

# BESOIN 3 : ARCHITECTURE DE CLASSIFICATION ET SAUVEGARDE DES IMAGES PNG

### 1. Importation des bibliothèques et configuration

Import des bibliothèques nécessaires :
- `pandas` / `numpy` : manipulation des données
- `joblib` : sérialisation du scaler et du modèle entraînés
- `matplotlib` / `seaborn` : visualisations statiques
- `scikit-learn` : `train_test_split`, `GridSearchCV`, `StandardScaler`, `LogisticRegression` et les métriques d'évaluation

In [1]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

sns.set_theme(style="whitegrid")

## 2. Préparation et nettoyage des données

**Variable cible :** `implantation`.

**Variables retenues :** `puissance`, `nb_pdc`, `gratuit`, `prise_ccs`, `prise_type2`, `prise_chademo`.

**Justification du choix des variables :**
- `puissance` et `nb_pdc` : caractéristiques techniques de la borne, corrélées au type de lieu où elle est installée (ex. puissance élevée plutôt sur station dédiée).
- `gratuit`, `prise_ccs`, `prise_type2`, `prise_chademo` : indicateurs booléens du type d'équipement, qui varient selon le contexte d'implantation (ex. CCS plus fréquent sur station de recharge rapide que sur parking privé).

**Justification du nettoyage :** suppression des lignes avec valeurs manquantes (`dropna`) sur ces variables et la cible uniquement — une ligne incomplète sur l'une d'elles est inutilisable pour l'entraînement.

In [2]:
df = pd.read_csv("export_IA.csv", low_memory=False)

colonne_cible = 'implantation'
colonnes_features = ['puissance', 'nb_pdc', 'gratuit', 'prise_ccs', 'prise_type2', 'prise_chademo']

df_propre = df.dropna(subset=colonnes_features + [colonne_cible]).copy()

### 2.1 Encodage des variables catégorielles

**Justification de l'encodage manuel des booléens :**
- Les colonnes `gratuit`, `prise_ccs`, `prise_type2`, `prise_chademo` arrivent sous des formats hétérogènes dans le CSV (`TRUE`/`FALSE`, `0`/`1`, parfois `OUI`/`NON`). Un mapping explicite vers `0`/`1` évite de dépendre du typage implicite de pandas, qui varie selon les valeurs réellement présentes.
- Les valeurs non reconnues sont mises à `0` (`fillna(0)`) plutôt que supprimées, car elles restent minoritaires et ne justifient pas de perdre la ligne entière.

In [3]:
colonnes_bool = ['gratuit', 'prise_ccs', 'prise_type2', 'prise_chademo']
for col in colonnes_bool:
    df_propre[col] = df_propre[col].astype(str).str.upper()
    df_propre[col] = df_propre[col].map({'TRUE': 1, 'FALSE': 0, '1': 1, '0': 0, 'OUI': 1, 'NON': 0}).fillna(0).astype(int)

## 3. Analyse Exploratoire des Données (Visualisations)

**Justification du choix des graphiques :**
- **Boxplot puissance par implantation** : permet de visualiser si la distribution de `puissance` diffère selon le type de lieu — une variable qui ne discrimine pas les classes n'apporterait rien au modèle.
- **Barplot proportion de prise CCS par implantation** : la prise CCS est un standard de charge rapide, sa fréquence devrait varier fortement entre une station dédiée et un parking résidentiel — ce graphique valide cette hypothèse avant de l'utiliser comme feature.

In [4]:
print("Enregistrement des graphiques de justification...")

plt.figure(figsize=(10, 5))
sns.boxplot(data=df_propre, x=colonne_cible, y='puissance', hue=colonne_cible, legend=False, palette='Set2')
plt.title("Justification de la variable 'puissance' selon l'Implantation", fontsize=12, fontweight='bold')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("output/justification_puissance.png", dpi=300)
plt.close()

plt.figure(figsize=(10, 5))
sns.barplot(data=df_propre, x=colonne_cible, y='prise_ccs', errorbar=None, hue=colonne_cible, legend=False, palette='Blues_r')
plt.title("Proportion de présence de Prises CCS par Implantation", fontsize=12, fontweight='bold')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("output/proportion_prise_ccs.png", dpi=300)
plt.close()

Enregistrement des graphiques de justification...


# 4. Prétraitement des données (Preprocessing)

## 4.1 Séparation du jeu de données (Train/Test Split)

**Justification des paramètres :**
- `test_size=0.2` : répartition standard 80/20, suffisamment de données d'entraînement tout en gardant un jeu de test représentatif.
- `stratify=y` : conserve la proportion de chaque classe d'implantation dans les jeux d'entraînement et de test, important ici car les classes sont déséquilibrées (`Voirie` largement majoritaire face à `Parking privé réservé à la clientèle`).

In [5]:
X = df_propre[colonnes_features].astype(float)
y = df_propre[colonne_cible]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 4.2 Normalisation des fonctionnalités et sauvegarde du Scaler

**Justification du choix de StandardScaler :**
- `puissance` et `nb_pdc` ont des échelles très différentes des variables booléennes (0/1) — sans normalisation, `LogisticRegression` accorderait un poids disproportionné aux variables à grande échelle.
- Le scaler est ajusté (`fit`) uniquement sur le jeu d'entraînement puis appliqué (`transform`) au jeu de test, pour éviter toute fuite d'information entre les deux.
- Le scaler est sauvegardé (`joblib`) car `main.py` doit appliquer exactement la même transformation aux nouvelles données, sans la réajuster.

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, 'scaler_pretraitement_b3.pkl')

['scaler_pretraitement_b3.pkl']

# 5. Entraînement du modèle et optimisation (GridSearchCV)

**Algorithme choisi : Régression Logistique**

**Justification du choix de l'algorithme :**
- Modèle linéaire simple et rapide à entraîner, donnant une baseline interprétable pour une classification multi-classes (5 types d'implantation).
- `GridSearchCV` (`cv=3`) teste plusieurs valeurs de régularisation `C` et de solveurs (`lbfgs`, `saga`) pour retenir la combinaison qui généralise le mieux, plutôt que de fixer des hyperparamètres arbitraires.

In [7]:
print("Recherche des meilleurs hyperparamètres (Régression Logistique)...")

param_grid = {
    'C': [0.1, 1.0, 10.0],
    'solver': ['lbfgs', 'saga'],
    'max_iter': [200]
}

modele_base = LogisticRegression(random_state=42)

grid_search = GridSearchCV(estimator=modele_base, param_grid=param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

meilleur_modele = grid_search.best_estimator_
print(f"Meilleurs hyperparamètres : {grid_search.best_params_}")

Recherche des meilleurs hyperparamètres (Régression Logistique)...


Meilleurs hyperparamètres : {'C': 0.1, 'max_iter': 200, 'solver': 'lbfgs'}


# 6. Évaluation du modèle final

**Justification du choix des métriques :**
- `classification_report` donne précision, rappel et F1-score par classe — indispensable ici car les classes sont déséquilibrées : l'accuracy globale seule masquerait une mauvaise prédiction sur les classes minoritaires.

In [8]:
y_pred = meilleur_modele.predict(X_test_scaled)

print("RAPPORT D'ÉVALUATION DE LA CLASSIFICATION")
print(classification_report(y_test, y_pred))

RAPPORT D'ÉVALUATION DE LA CLASSIFICATION
                                      precision    recall  f1-score   support

Parking privé réservé à la clientèle       1.00      0.32      0.48       307
        Parking privé à usage public       0.48      0.37      0.42      7499
                      Parking public       0.60      0.04      0.07      4110
 Station dédiée à la recharge rapide       0.58      0.38      0.46      1478
                              Voirie       0.51      0.86      0.64      9185

                            accuracy                           0.51     22579
                           macro avg       0.63      0.39      0.41     22579
                        weighted avg       0.53      0.51      0.45     22579



## 6.1 Matrice de confusion

**Justification :** complète le rapport de classification en visualisant précisément quelles classes sont confondues entre elles — utile ici pour identifier que `Parking public` est largement confondu avec `Voirie` (cf. discussion plus bas).

In [9]:
matrice = confusion_matrix(y_test, y_pred)
classes = meilleur_modele.classes_

plt.figure(figsize=(8, 6))
sns.heatmap(matrice, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title("Matrice de Confusion - Implantation de la Station", fontsize=12, fontweight='bold')
plt.ylabel('Réalité Terrain')
plt.xlabel('Prédiction Modèle')
plt.tight_layout()
plt.savefig("output/matrice_confusion.png", dpi=300)
plt.close()

# 7. Exportation du modèle prédictif final

**Justification :** le modèle est sauvegardé (`joblib`) pour permettre à `main.py` de charger directement le modèle entraîné sans jamais relancer d'apprentissage à l'exécution.

In [10]:
joblib.dump(meilleur_modele, 'modele_classification_b3.pkl')


['modele_classification_b3.pkl']